In [1]:
import pandas as pd

df = pd.read_csv("dream_home_form_sample.csv")  # put your exact file name here
print("Rows:", len(df))
print(df.head())


Rows: 30
   sqft  bedrooms  bathrooms  age       city        state  zipcode  \
0   804         6        2.0   31  Hyderabad    Telangana   500001   
1  4056         1        1.0   11  Bangalore    Karnataka   560001   
2  4036         2        2.5   75       Pune  Maharashtra   411001   
3  3387         3        1.5   27    Houston        Texas    77001   
4  1392         3        4.0   44    Chicago     Illinois    60601   

  property_type             features   design_style  
0         condo                 pool      craftsman  
1         condo               garage      craftsman  
2     townhouse                  NaN    traditional  
3     townhouse                  NaN         modern  
4     apartment  garage|master_suite  mediterranean  


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import joblib

# -------------------------
# 1. Generate dataset
# -------------------------
np.random.seed(42)
n = 500
sqft = np.random.randint(500, 4000, n)
rooms = np.random.randint(1, 6, n)
bathrooms = np.random.randint(1, 4, n)
age = np.random.randint(0, 50, n)
location = np.random.choice(["Downtown", "Suburb", "Countryside"], n)

price = (
    sqft * np.random.uniform(200, 300) +
    rooms * 50000 +
    bathrooms * 30000 -
    age * 1000 +
    np.where(location == "Downtown", 150000,
             np.where(location == "Suburb", 80000, 30000)) +
    np.random.normal(0, 50000, n)
)

df = pd.DataFrame({
    "sqft": sqft,
    "rooms": rooms,
    "bathrooms": bathrooms,
    "age": age,
    "location": location,
    "price": price.astype(int)
})

df.to_csv("housing.csv", index=False)
print("✅ Dataset saved as housing.csv")

# -------------------------
# 2. Train Model
# -------------------------
X = df.drop("price", axis=1)
y = df["price"]

numeric_features = ["sqft", "rooms", "bathrooms", "age"]
categorical_features = ["location"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(), categorical_features)
])

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)

# -------------------------
# 3. Save Preprocessor & Model
# -------------------------
joblib.dump(preprocessor, "preprocessor.joblib")
joblib.dump(model, "xgb_model.joblib")

print("✅ Model and preprocessor saved!")


✅ Dataset saved as housing.csv
✅ Model and preprocessor saved!
